In [1]:
import sys
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

Python: 3.13.18
pandas: 2.2.3
numpy: 2.1.3
scikit-learn: 1.6.1
joblib: 1.6.0


In [2]:
from google.colab import files
uploaded = files.upload()

Saving insurance.csv to insurance (1).csv


In [3]:
df = pd.read_csv("insurance.csv")

print("Shape (rows, columns):", df.shape)
print()
df.head()

Shape (rows, columns): (1338, 7)



,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
print("=== Data types and non-null counts ===")
df.info()

print("\n=== Missing values per column ===")
print(df.isnull().sum())

print("\n=== Duplicate rows ===")
print(df.duplicated().sum())

print("\n=== Summary statistics ===")
print(df.describe())

=== Data types and non-null counts ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB

=== Missing values per column ===
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

=== Duplicate rows ===
1

=== Summary statistics ===
               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397     1.094918  13270.422265
std      14.049960     6.098187     1.205493  12110.011237
min  

In [5]:
print("=== Target: charges ===")
print(df["charges"].describe())
print("Skewness:", round(df["charges"].skew(), 2))

print("\n=== Category values (we need these exact spellings for the API later) ===")
for col in ["sex", "smoker", "region"]:
    print(col, "->", sorted(df[col].unique()))

print("\n=== Average charges by smoker ===")
print(df.groupby("smoker")["charges"].mean().round(2))

=== Target: charges ===
count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010
Name: charges, dtype: float64
Skewness: 1.52

=== Category values (we need these exact spellings for the API later) ===
sex -> ['female', 'male']
smoker -> ['no', 'yes']
region -> ['northeast', 'northwest', 'southeast', 'southwest']

=== Average charges by smoker ===
smoker
no      8434.27
yes    32050.23
Name: charges, dtype: float64


In [6]:
# 1. Remove duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate row(s). Rows now: {len(df)}")

# 2. Separate features (X) from target (y)
numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]
feature_columns = numeric_features + categorical_features

X = df[feature_columns]
y = df["charges"]

# 3. Train/test split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

Removed 1 duplicate row(s). Rows now: 1337
X_train: (1069, 6) | X_test: (268, 6)


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=3,
            random_state=42,
        )),
    ]
)

model

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sex', 'smoker', 'region']),
                                                 ('num', 'passthrough',
                                                  ['age', 'bmi',
                                                   'children'])])),
                ('regressor',
                 RandomForestRegressor(min_samples_leaf=3, n_estimators=200,
                                       random_state=42))])

In [8]:
model.fit(X_train, y_train)
print("Training finished.")

y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# First 5 predictions vs actual values
comparison = pd.DataFrame({
    "actual": y_test.values[:5],
    "predicted": y_pred_test[:5].round(2),
})
comparison

Training finished.


,actual,predicted
0,8688.85885,9852.99
1,5708.86700,6283.49
2,11436.73815,12266.09
3,38746.35510,39827.81
4,4463.20510,5527.94


In [9]:
def evaluate(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return round(r2, 4), round(mae, 2), round(rmse, 2)

# Our Random Forest
rf_train = evaluate(y_train, y_pred_train)
rf_test = evaluate(y_test, y_pred_test)

# Simple baseline: Linear Regression with the same preprocessing
baseline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", LinearRegression()),
])
baseline.fit(X_train, y_train)
lr_test = evaluate(y_test, baseline.predict(X_test))

results = pd.DataFrame(
    [rf_train, rf_test, lr_test],
    columns=["R2", "MAE", "RMSE"],
    index=["RandomForest (train)", "RandomForest (test)", "LinearRegression (test)"],
)
results

,R2,MAE,RMSE
RandomForest (train),0.9227,1762.82,3252.27
RandomForest (test),0.8961,2417.21,4369.39
LinearRegression (test),0.8069,4177.05,5956.34


In [10]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
importances = model.named_steps["regressor"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
importance_df

,feature,importance
0,cat__smoker_no,0.487252
1,num__bmi,0.192634
2,cat__smoker_yes,0.153957
3,num__age,0.133747
4,num__children,0.016455
5,cat__region_northwest,0.003639
6,cat__region_northeast,0.003624
7,cat__region_southeast,0.002729
8,cat__region_southwest,0.002120
9,cat__sex_female,0.002038


In [11]:
# 1. Save the full pipeline (preprocessing + model together)
joblib.dump(model, "model.joblib")

# 2. Save helpful info for the API (built only from real values above)
model_info = {
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "categories": {col: sorted(df[col].unique().tolist()) for col in categorical_features},
    "target": "charges",
    "sklearn_version": sklearn.__version__,
    "python_version": sys.version.split()[0],
    "test_metrics": {
        "r2": float(rf_test[0]),
        "mae": float(rf_test[1]),
        "rmse": float(rf_test[2]),
    },
}

with open("model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print(json.dumps(model_info, indent=2))

{
  "feature_columns": [
    "age",
    "bmi",
    "children",
    "sex",
    "smoker",
    "region"
  ],
  "numeric_features": [
    "age",
    "bmi",
    "children"
  ],
  "categorical_features": [
    "sex",
    "smoker",
    "region"
  ],
  "categories": {
    "sex": [
      "female",
      "male"
    ],
    "smoker": [
      "no",
      "yes"
    ],
    "region": [
      "northeast",
      "northwest",
      "southeast",
      "southwest"
    ]
  },
  "target": "charges",
  "sklearn_version": "1.6.1",
  "python_version": "3.13.18",
  "test_metrics": {
    "r2": 0.8961,
    "mae": 2417.21,
    "rmse": 4369.39
  }
}


In [12]:
loaded_model = joblib.load("model.joblib")

sample = pd.DataFrame([{
    "age": 35,
    "bmi": 27.5,
    "children": 2,
    "sex": model_info["categories"]["sex"][0],
    "smoker": model_info["categories"]["smoker"][0],
    "region": model_info["categories"]["region"][0],
}])

print("Sample input:")
print(sample)
print("\nPredicted charge:", round(float(loaded_model.predict(sample)[0]), 2))

Sample input:
   age   bmi  children     sex smoker     region
0   35  27.5         2  female     no  northeast

Predicted charge: 7102.92


In [13]:
files.download("model.joblib")
files.download("model_info.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>